# 🚀 Welcome to MalthusJAX: 5-Minute Quickstart

**MalthusJAX** is a high-performance evolutionary computation library written in pure JAX. It compiles entire evolutionary runs end-to-end into hardware-accelerated XLA graphs via `jax.lax.scan`—running at thousands of generations per second with zero Python loop overhead.

### 🧭 How is MalthusJAX structured?
MalthusJAX has **4 levels of abstraction** so you never have to write low-level code unless you want to:
- **Level 4 (Composer)**: ⭐ **Start here!** Declarative, zero-boilerplate API to configure, run, and compare algorithms.
- **Level 3 (Engines)**: The 5-phase evolutionary loop (`lax.scan`, state PyTrees).
- **Level 2 (Operators)**: Vectorized, GPU-accelerated mutation, crossover, and selection operators.
- **Level 1 (Core)**: Foundational PyTree data structures (`BaseGenome`, `BasePopulation`, `ComposableEvaluator`).

In [ ]:
import malthusjax as mj
import matplotlib.pyplot as plt

print(f"✓ MalthusJAX Version: {mj.__version__}")

## 1️⃣ Run Your First Genetic Algorithm in 3 Lines

Let's optimize a 10-dimensional **Sphere** benchmark function ($f(\mathbf{x}) = -\sum x_i^2$).
All we need is the `Composer`:

In [ ]:
# 1. Create a default Composer
composer = mj.Composer.create_default()

# 2. Run a Genetic Algorithm across 3 random seeds
result = composer.quick_run(
    fitness="sphere:dim=10",
    pop_size=64,
    generations=100,
    seeds=(42, 43, 44),
)

# 3. Print the aggregated statistical summary
summary = result.aggregated_summary()
print("✓ Optimization Finished!")
print(f"  Best Fitness (Mean): {summary['best_fitness']['mean']:.4e}")
print(f"  Total Evaluations:   {summary['total_evaluations']['mean']:.0f}")
print(f"  Mean Duration:       {summary['duration_seconds']['mean']:.3f}s")

## 2️⃣ Inspecting Convergence & History

Every run produces rich telemetry, including generation-by-generation best and mean fitness values.

In [ ]:
# Access the first seed's run result
first_run = result.runs[0]
print(f"Seed {first_run.seed} Best Fitness: {first_run.metrics['best_fitness']:.4e}")

# Extract generations and fitness trajectory
generations = [h["generation"] for h in first_run.history]
best_fitness = [h["best_fitness"] for h in first_run.history]
mean_fitness = [h["mean_fitness"] for h in first_run.history]

# Plot convergence curve
plt.figure(figsize=(8, 4))
plt.plot(generations, best_fitness, color="#2563EB", lw=2, label=f"Best (Seed {first_run.seed})")
plt.plot(generations, mean_fitness, color="#93C5FD", lw=1.5, linestyle="--", label="Population Mean")
plt.xlabel("Generation", fontsize=11)
plt.ylabel("Fitness (Closer to 0 is better)", fontsize=11)
plt.title("MalthusJAX GA Convergence (Sphere 10D)", fontsize=13, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 3️⃣ Customizing Genetic Operators Declaratively

MalthusJAX features a built-in catalog of operators that can be configured with string parameters.
Here, we solve the multimodal **Rastrigin** function using **Tournament Selection**, **Blend Crossover**, and **Gaussian Mutation**:

In [ ]:
rastrigin_result = composer.quick_run(
    fitness="rastrigin:dim=10",
    selection="tournament:tournament_size=4",
    crossover="blend:alpha=0.5",
    mutation="gaussian:mutation_rate=0.15,mutation_strength=0.1",
    pop_size=100,
    generations=150,
    seeds=(42,),
)

r_summary = rastrigin_result.aggregated_summary()
print(f"✓ Rastrigin Best Fitness: {r_summary['best_fitness']['mean']:.4e}")

## 4️⃣ Benchmarking Algorithm Variants (`composer.compare`)

Want to empirically compare Tournament Selection vs Roulette Selection?
`composer.compare()` executes multiple algorithmic variants with **identical initial populations and seeds** for statistically rigorous comparison:

In [ ]:
comparison = composer.compare(
    pipelines={
        "GA + Tournament": dict(selection="tournament:tournament_size=3"),
        "GA + Roulette": dict(selection="roulette"),
    },
    fitness="sphere:dim=10",
    pop_size=64,
    generations=50,
    seeds=(10, 20, 30),
)

# Print formatted comparison summary
table = comparison.summary_table()
for name, metrics in table.items():
    print(f"{name:16} | Best: {metrics['best_fitness']['mean']:10.4f} | Time: {metrics['duration_seconds']['mean']:.2f}s")

# Plot comparative convergence
fig, ax = plt.subplots(figsize=(8, 4))
comparison.plot_convergence(seed_index=0, ax=ax)
ax.set_title("Tournament vs Roulette Convergence (Seed 10)")
plt.tight_layout()
plt.show()

---

## 🗺️ Next Steps: Choosing Your Abstraction Level

Depending on what you are building, pick your level:

| Goal | Recommended Path | Guide / Notebook |
| :--- | :--- | :--- |
| **Run sweeps & experiments** | **Level 4: Composer** | [`composer_technical_overview.ipynb`](composer_technical_overview.ipynb) / [`level_4_composer_guide.md`](../docs/framework_guides/level_4_composer_guide.md) |
| **Customize evolutionary loops** | **Level 3: Engines** | [`engine_technical_overview.ipynb`](engine_technical_overview.ipynb) / [`level_3_engines_guide.md`](../docs/framework_guides/level_3_engines_guide.md) |
| **Vectorized genetic operators** | **Level 2: Operators** | [`crossover_technical_overview.ipynb`](crossover_technical_overview.ipynb) / [`mutation_technical_overview.ipynb`](mutation_technical_overview.ipynb) |
| **Novel PyTrees / Composable Fitness** | **Level 1: Core** | [`genome_technical_overview.ipynb`](genome_technical_overview.ipynb) / [`level_1_core_guide.md`](../docs/framework_guides/level_1_core_guide.md) |
| **Scaffold new components** | **CLI Tool** | Run `make scaffold` or `python scripts/scaffold.py -h` |